# EGF Batch Processing Pipeline

**Goal:** 3D segmentation of whole cells and intracellular signal blobs, then calculate volumetric overlap between signal channels.

| Index | Channel | Segmentation |
|-------|---------|-------------|
| Ch0 | Signal blobs | Cellpose (whole-cell mask) → 3D blob segmentation |
| Ch1 | Signal blobs | 3D blob segmentation |
| Ch2 | Signal blobs | 3D blob segmentation |
| Ch3 | Nucleus | 3D blob segmentation (no overlap analysis) |

**Overlaps to compute per cell:**
- Ch0 blobs ∩ Ch1 blobs
- Ch0 blobs ∩ Ch2 blobs
- Ch1 blobs ∩ Ch2 blobs

In [58]:
import numpy as np
import pandas as pd
import tifffile
from pathlib import Path
from scipy import ndimage as ndi
from skimage import filters, morphology, measure, segmentation

# ── Voxel dimensions (µm) — used throughout the notebook ─────────────────────
XY_PIXEL_UM = 0.064   # µm per XY pixel
Z_STEP_UM   = 0.130   # µm per Z step

## 1. Data Directory

In [59]:
DATA_DIR = Path(r"Z:\Marta\20260218\2026-02-18-Decon")

# Collect all OME-TIFF files, sorted by name
tiff_files = sorted(DATA_DIR.glob("*.ome.tiff"))

print(f"Found {len(tiff_files)} files in {DATA_DIR}\n")
for f in tiff_files:
    print(f.name)

Found 63 files in Z:\Marta\20260218\2026-02-18-Decon

H23 607-ko EGF 30min_10_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_1_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_3_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_4_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_5_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_6_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_7_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_8_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 30min_9_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_10_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_11_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_2_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_3_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_4_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_5_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_6_MMStack_Pos0.ome_cmle.ome.tiff
H23 607-ko EGF 5min_7_MMStack_Po

## 2. Load a Single Image for Development

Pick one file to work with interactively before running the full batch.

In [61]:
# Change the index to try a different file
test_file = tiff_files[2]
print("Loading:", test_file.name)

with tifffile.TiffFile(test_file) as tif:
    img = tif.asarray()   # shape: (C, Z, Y, X)

print(f"Shape: {img.shape}  |  dtype: {img.dtype}")
print(f"Channels: Ch0 min={img[0].min():.1f} max={img[0].max():.1f}")
print(f"          Ch1 min={img[1].min():.1f} max={img[1].max():.1f}")
print(f"          Ch2 min={img[2].min():.1f} max={img[2].max():.1f}")
print(f"          Ch3 min={img[3].min():.1f} max={img[3].max():.1f}")

# Split into named channel arrays for convenience
ch0, ch1, ch2, ch3 = img[0], img[1], img[2], img[3]

Loading: H23 607-ko EGF 30min_2_MMStack_Pos0.ome_cmle.ome.tiff
Shape: (4, 54, 2048, 2048)  |  dtype: float32
Channels: Ch0 min=0.0 max=14497.7
          Ch1 min=0.0 max=3196.7
          Ch2 min=0.0 max=38252.0
          Ch3 min=0.0 max=629388.6


### Napari — raw channels

In [62]:
import napari

viewer = napari.Viewer(title=test_file.name)

viewer.add_image(ch0, name="Ch0 (signal)", colormap="green",  blending="additive")
viewer.add_image(ch1, name="Ch1 (signal)", colormap="magenta", blending="additive")
viewer.add_image(ch2, name="Ch2 (signal)", colormap="cyan",   blending="additive")
viewer.add_image(ch3, name="Ch3 (nucleus)", colormap="blue",  blending="additive")

<Image layer 'Ch3 (nucleus)' at 0x1bac30b9f10>

### Cellpose smoke test — run this first

In [63]:
from cellpose import models
import numpy as np

# Tiny synthetic image — 5 slices, 64x64, single channel (Z, Y, X)
test_img = np.random.randint(0, 65535, (5, 64, 64), dtype=np.uint16).astype(np.float32) / 65535.0

cp_model = models.CellposeModel(gpu=True, pretrained_model="cpsam")
masks, _, _ = cp_model.eval(test_img, z_axis=0, do_3D=False, stitch_threshold=0.5)

print("Smoke test passed — Cellpose is working")
print("Output mask shape:", masks.shape)

100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 4/4 [00:00<?, ?it/s]

Smoke test passed â€” Cellpose is working
Output mask shape: (5, 64, 64)


In [64]:
from cellpose import models
from scipy.ndimage import gaussian_filter
import numpy as np

# --- parameters to tune ---
CELLPOSE_MODEL   = "cpsam"  # default model in Cellpose v4
CELL_DIAMETER    = 250    # None = auto-estimate; set in pixels (XY) if auto is off
STITCH_THRESHOLD = 0.5      # links 2D masks across Z slices (0–1; lower = more linking)
ANISOTROPY       = 2.03      # Z_voxel_size / XY_voxel_size — adjust to your voxel dims
BLUR_SIGMA       = 10      # Gaussian blur applied to Ch0 before segmentation —
                            # smears punctate dots into a smooth cell-body signal;
                            # increase if cells still fragment, decrease if they merge
# --------------------------

def norm_float(arr):
    arr = arr.astype(np.float32)
    arr -= arr.min()
    if arr.max() > 0:
        arr /= arr.max()
    return arr

# Blur Ch0 to turn punctate signal into a smooth cell envelope
ch0_blurred = gaussian_filter(ch0.astype(np.float32), sigma=BLUR_SIGMA)
ch0_norm    = norm_float(ch0_blurred)

# Ch3 = nucleus — gives Cellpose a clean anchor for cell boundaries
ch3_norm = norm_float(ch3)

# Stack as (Z, Y, X, 2): channel 0 = cytoplasm (blurred Ch0), channel 1 = nucleus (Ch3)
cyto_nuc = np.stack([ch0_norm, ch3_norm], axis=-1)   # shape: (Z, Y, X, 2)

cp_model = models.CellposeModel(gpu=True, pretrained_model=CELLPOSE_MODEL)

cell_masks, flows, styles = cp_model.eval(
    cyto_nuc,
    diameter=CELL_DIAMETER,
    z_axis=0,
    channel_axis=3,             # last axis holds the 2 channels
    do_3D=False,
    stitch_threshold=STITCH_THRESHOLD,
    anisotropy=ANISOTROPY,
)

n_cells = cell_masks.max()
print(f"Detected {n_cells} cell(s)")
print(f"Mask shape: {cell_masks.shape}  |  dtype: {cell_masks.dtype}")

100%|â–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆâ–ˆ| 53/53 [00:04<00:00, 11.52it/s]


Detected 29 cell(s)
Mask shape: (54, 2048, 2048)  |  dtype: uint16


### Napari — whole-cell masks

In [65]:
viewer.add_labels(cell_masks, name="Cellpose whole-cell")

<Labels layer 'Cellpose whole-cell' at 0x1b6016dfad0>

## 4. Pick One Cell to Develop Blob Segmentation

Isolate a single cell by its label ID so we can tune blob parameters on a manageable volume.

In [70]:
CELL_ID = 1   # change to whichever cell label you want to inspect

single_cell_mask = (cell_masks == CELL_ID)

# find_objects needs the integer label array, returns one slice per label (0-indexed)
bbox = ndi.find_objects(cell_masks)[CELL_ID - 1]

mask_crop = single_cell_mask[bbox]
ch0_crop  = ch0[bbox]
ch1_crop  = ch1[bbox]
ch2_crop  = ch2[bbox]
ch3_crop  = ch3[bbox]

print(f"Bounding box: {bbox}")
print(f"Crop shape:   {ch0_crop.shape}")

Bounding box: (slice(0, 29, None), slice(0, 171, None), slice(471, 757, None))
Crop shape:   (29, 171, 286)


## 5. 3D Blob Segmentation (Ch0, Ch1, Ch2)

Approach: Gaussian smooth → threshold (Li method, works well for blobs) → remove small objects → label connected components.  
Key parameters to tune:
- `SIGMA` — smoothing scale (in voxels); increase if blobs merge, decrease if they fragment  
- `MIN_BLOB_VOXELS` — minimum blob size to keep (filters noise spots)

In [71]:
from skimage import feature, segmentation

def segment_blobs_3d(channel_crop, cell_mask,
                     sigma_small=1.0, sigma_large=3.0,
                     dog_thresh_rel=0.1, min_voxels=50, min_distance=3):
    """
    DoG + watershed blob segmentation.

    Parameters
    ----------
    channel_crop  : ndarray (Z, Y, X)
    cell_mask     : bool ndarray — True inside the cell
    sigma_small   : float — inner Gaussian sigma (voxels); sets minimum blob size
    sigma_large   : float — outer Gaussian sigma (voxels); sets maximum blob size
    dog_thresh_rel: float — threshold as fraction of DoG max (0–1);
                            uniform across dim and bright blobs — tune this first
    min_voxels    : int   — discard blobs smaller than this after segmentation
    min_distance  : int   — minimum voxel distance between blob centres

    Returns
    -------
    labels : int ndarray — labelled blobs (0 = background)
    """
    # Normalise to 0–1
    arr = channel_crop.astype(np.float32)
    arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8)

    # DoG — enhances blobs between sigma_small and sigma_large regardless of brightness
    smooth_s = filters.gaussian(arr,   sigma=sigma_small)
    smooth_l = filters.gaussian(arr,   sigma=sigma_large)
    dog      = smooth_s - smooth_l
    dog[~cell_mask] = 0

    # Threshold on DoG response (relative) — dim and bright blobs treated equally
    binary = dog > (dog.max() * dog_thresh_rel)
    binary = morphology.remove_small_objects(binary, max_size=min_voxels)

    # Local maxima in smoothed image as watershed seeds
    coords = feature.peak_local_max(
        smooth_s * cell_mask,
        min_distance=min_distance,
        labels=binary.astype(np.int32)
    )
    seed_mask = np.zeros_like(dog, dtype=bool)
    if len(coords):
        seed_mask[tuple(coords.T)] = True
    seeds = measure.label(seed_mask)

    # Watershed — splits touching blobs, tight to real signal
    labels = segmentation.watershed(-smooth_s, seeds, mask=binary)
    labels = measure.label(labels > 0)   # relabel cleanly after any removals

    return labels


# --- parameters to tune ---
SIGMA_SMALL    = .7    # voxels — min blob scale (~half the smallest blob radius)
SIGMA_LARGE    = 3.0    # voxels — max blob scale (~half the largest blob radius)
DOG_THRESH_REL = 0.05  # 0–1 fraction of DoG max; lower = more/dimmer blobs included
MIN_BLOB_VOXELS = 50    # discard anything smaller than this
MIN_DISTANCE   = 2      # min voxels between blob centres (prevents over-splitting)
# --------------------------

blobs_ch0 = segment_blobs_3d(ch0_crop, mask_crop,
                              sigma_small=SIGMA_SMALL, sigma_large=SIGMA_LARGE,
                              dog_thresh_rel=DOG_THRESH_REL, min_voxels=MIN_BLOB_VOXELS,
                              min_distance=MIN_DISTANCE)
blobs_ch1 = segment_blobs_3d(ch1_crop, mask_crop,
                              sigma_small=SIGMA_SMALL, sigma_large=SIGMA_LARGE,
                              dog_thresh_rel=DOG_THRESH_REL, min_voxels=MIN_BLOB_VOXELS,
                              min_distance=MIN_DISTANCE)
blobs_ch2 = segment_blobs_3d(ch2_crop, mask_crop,
                              sigma_small=SIGMA_SMALL, sigma_large=SIGMA_LARGE,
                              dog_thresh_rel=DOG_THRESH_REL, min_voxels=MIN_BLOB_VOXELS,
                              min_distance=MIN_DISTANCE)

print(f"Ch0 blobs: {blobs_ch0.max()}")
print(f"Ch1 blobs: {blobs_ch1.max()}")
print(f"Ch2 blobs: {blobs_ch2.max()}")

Ch0 blobs: 30
Ch1 blobs: 55
Ch2 blobs: 30


### Napari — blob segmentations on single cell

In [72]:
viewer2 = napari.Viewer(title=f"Cell {CELL_ID} — blob segmentation")

viewer2.add_image(ch0_crop, name="Ch0 (signal)", colormap="green",  blending="additive")
viewer2.add_image(ch1_crop, name="Ch1 (signal)", colormap="magenta", blending="additive")
viewer2.add_image(ch2_crop, name="Ch2 (signal)", colormap="cyan",   blending="additive")

viewer2.add_labels(blobs_ch0, name="Blobs Ch0", opacity=0.6)
viewer2.add_labels(blobs_ch1, name="Blobs Ch1", opacity=0.6)
viewer2.add_labels(blobs_ch2, name="Blobs Ch2", opacity=0.6)

<Labels layer 'Blobs Ch2' at 0x1b83ca49d10>

## 6. Measurements

For every blob in Ch0, Ch1, and Ch2 we record:

| Group | Columns |
|---|---|
| Identity | `file`, `cell_id`, `channel`, `blob_id` |
| Volume | `volume_voxels`, `volume_um3` |
| Intensity (mean + sum) | `ch0/1/2_mean_intensity`, `ch0/1/2_sum_intensity` |
| Position | `centroid_z/y/x_px` |
| Distance | `dist_to_surface_um`, `dist_to_nucleus_um` |
| Overlap (per cell) | `c0_c1_overlap_voxels`, `c0_c2_overlap_voxels`, `c1_c2_overlap_voxels` |

Rows are grouped by channel (C0 → C1 → C2) so the CSV can be filtered or pivoted easily.

In [ ]:
import pandas as pd

# ── Voxel metrics ─────────────────────────────────────────────────────────────
BLOB_METRICS = [
    'volume_voxels', 'volume_um3',
    'dist_to_surface_um', 'dist_to_nucleus_um',
    'ch0_mean_intensity', 'ch1_mean_intensity', 'ch2_mean_intensity',
    'ch0_sum_intensity',  'ch1_sum_intensity',  'ch2_sum_intensity',
]
STAT_FUNCS = ['mean', 'std', 'min', 'max', 'median']

# ── Nucleus centroid ──────────────────────────────────────────────────────────
def get_nucleus_centroid_um(ch3_crop, mask_crop,
                             voxel_size=(Z_STEP_UM, XY_PIXEL_UM, XY_PIXEL_UM),
                             sigma=2.0):
    """
    Threshold Ch3 (nucleus) inside the cell and return the centroid in µm.
    Returns None if the channel is empty.
    """
    vz, vxy, _ = voxel_size
    arr = ch3_crop.astype(np.float32)
    rng = arr.max() - arr.min()
    if rng == 0:
        return None
    arr = (arr - arr.min()) / rng
    smoothed = filters.gaussian(arr, sigma=sigma)
    thresh = filters.threshold_otsu(smoothed[mask_crop])
    nuc_bin = (smoothed > thresh) & mask_crop
    if nuc_bin.sum() == 0:
        return None
    cz, cy, cx = ndi.center_of_mass(nuc_bin)
    return np.array([cz * vz, cy * vxy, cx * vxy])  # µm


# ── Per-cell measurement ──────────────────────────────────────────────────────
def measure_cell(file_name, cell_id, cell_masks,
                 ch0, ch1, ch2, ch3,
                 seg_params,
                 voxel_size=(Z_STEP_UM, XY_PIXEL_UM, XY_PIXEL_UM)):
    """
    Crop one cell, segment blobs in Ch0/1/2, return one row per blob.
    Rows are grouped C0 → C1 → C2.
    """
    vz, vxy, _ = voxel_size
    voxel_vol_um3 = vz * vxy * vxy

    single_cell_mask = (cell_masks == cell_id)
    bbox = ndi.find_objects(cell_masks)[cell_id - 1]

    mask_crop = single_cell_mask[bbox]
    ch0_crop  = ch0[bbox];  ch1_crop = ch1[bbox]
    ch2_crop  = ch2[bbox];  ch3_crop = ch3[bbox]

    blobs_ch0 = segment_blobs_3d(ch0_crop, mask_crop, **seg_params)
    blobs_ch1 = segment_blobs_3d(ch1_crop, mask_crop, **seg_params)
    blobs_ch2 = segment_blobs_3d(ch2_crop, mask_crop, **seg_params)

    bin0 = blobs_ch0 > 0;  bin1 = blobs_ch1 > 0;  bin2 = blobs_ch2 > 0
    c0_c1_ov = int((bin0 & bin1).sum())
    c0_c2_ov = int((bin0 & bin2).sum())
    c1_c2_ov = int((bin1 & bin2).sum())

    dist_surface = ndi.distance_transform_edt(mask_crop, sampling=(vz, vxy, vxy))
    nuc_um = get_nucleus_centroid_um(ch3_crop, mask_crop, voxel_size)

    rows = []
    for ch_name, blobs in [('C0', blobs_ch0), ('C1', blobs_ch1), ('C2', blobs_ch2)]:
        for prop in measure.regionprops(blobs):
            blob_mask = blobs == prop.label
            cz, cy, cx = prop.centroid
            ci = (int(round(cz)), int(round(cy)), int(round(cx)))
            dist_surf = float(dist_surface[ci])
            dist_nuc  = float(np.linalg.norm(
                np.array([cz*vz, cy*vxy, cx*vxy]) - nuc_um
            )) if nuc_um is not None else np.nan

            rows.append({
                'file': file_name, 'cell_id': cell_id,
                'channel': ch_name, 'blob_id': prop.label,
                'volume_voxels':       prop.area,
                'volume_um3':          round(prop.area * voxel_vol_um3, 4),
                'centroid_z_px':       round(cz, 2),
                'centroid_y_px':       round(cy, 2),
                'centroid_x_px':       round(cx, 2),
                'ch0_mean_intensity':  round(float(ch0_crop[blob_mask].mean()), 4),
                'ch1_mean_intensity':  round(float(ch1_crop[blob_mask].mean()), 4),
                'ch2_mean_intensity':  round(float(ch2_crop[blob_mask].mean()), 4),
                'ch0_sum_intensity':   round(float(ch0_crop[blob_mask].sum()),  2),
                'ch1_sum_intensity':   round(float(ch1_crop[blob_mask].sum()),  2),
                'ch2_sum_intensity':   round(float(ch2_crop[blob_mask].sum()),  2),
                'dist_to_surface_um':  round(dist_surf, 4),
                'dist_to_nucleus_um':  round(dist_nuc, 4) if not np.isnan(dist_nuc) else np.nan,
                'c0_c1_overlap_voxels': c0_c1_ov,
                'c0_c2_overlap_voxels': c0_c2_ov,
                'c1_c2_overlap_voxels': c1_c2_ov,
            })
    return rows


# ── Summary: one row per cell ─────────────────────────────────────────────────
def make_summary_df(df):
    """
    Collapse per-blob DataFrame into one row per cell.
    Columns: file, cell_id, overlap counts, n_blobs per channel,
             mean/std/min/max/median of every metric per channel.
    """
    rows = []
    for (fname, cid), cell_df in df.groupby(['file', 'cell_id'], sort=False):
        row = {'file': fname, 'cell_id': cid}
        for col in ['c0_c1_overlap_voxels', 'c0_c2_overlap_voxels', 'c1_c2_overlap_voxels']:
            row[col] = int(cell_df[col].iloc[0])
        for ch in ['C0', 'C1', 'C2']:
            pfx   = ch.lower()
            ch_df = cell_df[cell_df['channel'] == ch]
            row[f'{pfx}_n_blobs'] = len(ch_df)
            for metric in BLOB_METRICS:
                vals = ch_df[metric].dropna()
                if vals.empty:
                    for s in STAT_FUNCS:
                        row[f'{pfx}_{metric}_{s}'] = np.nan
                else:
                    row[f'{pfx}_{metric}_mean']   = round(float(vals.mean()),   4)
                    row[f'{pfx}_{metric}_std']    = round(float(vals.std()),    4)
                    row[f'{pfx}_{metric}_min']    = round(float(vals.min()),    4)
                    row[f'{pfx}_{metric}_max']    = round(float(vals.max()),    4)
                    row[f'{pfx}_{metric}_median'] = round(float(vals.median()), 4)
        rows.append(row)
    return pd.DataFrame(rows)


# ── CSV writer ────────────────────────────────────────────────────────────────
def save_csvs(df_blobs, out_dir, stem):
    """Write granular (per-blob) and summary (per-cell) CSVs."""
    df_blobs = df_blobs.copy()
    df_blobs['channel'] = pd.Categorical(df_blobs['channel'], ['C0', 'C1', 'C2'])
    df_blobs = df_blobs.sort_values(['file', 'cell_id', 'channel', 'blob_id']).reset_index(drop=True)

    p_blobs   = out_dir / f"{stem}_blobs.csv"
    p_summary = out_dir / f"{stem}_cell_summary.csv"

    df_blobs.to_csv(p_blobs, index=False)
    df_summary = make_summary_df(df_blobs)
    df_summary.to_csv(p_summary, index=False)

    print(f"Granular CSV  ({len(df_blobs):,} rows)                       → {p_blobs.name}")
    print(f"Summary CSV   ({len(df_summary):,} rows, {len(df_summary.columns)} cols) → {p_summary.name}")
    return df_blobs, df_summary


print("measure_cell(), make_summary_df(), save_csvs() defined")

### 6a. Test measurements on the single loaded image

In [74]:
seg_params = dict(
    sigma_small    = SIGMA_SMALL,
    sigma_large    = SIGMA_LARGE,
    dog_thresh_rel = DOG_THRESH_REL,
    min_voxels     = MIN_BLOB_VOXELS,
    min_distance   = MIN_DISTANCE,
)

all_rows = []
n_cells = int(cell_masks.max())

for cid in range(1, n_cells + 1):
    print(f"  measuring cell {cid}/{n_cells} ...", end=" ", flush=True)
    rows = measure_cell(
        file_name  = test_file.name,
        cell_id    = cid,
        cell_masks = cell_masks,
        ch0=ch0, ch1=ch1, ch2=ch2, ch3=ch3,
        seg_params = seg_params,
    )
    all_rows.extend(rows)
    print(f"{len(rows)} blobs")

df_single = pd.DataFrame(all_rows)

# Sort so C0 rows come first, then C1, then C2
df_single['channel'] = pd.Categorical(df_single['channel'], ['C0', 'C1', 'C2'])
df_single = df_single.sort_values(['cell_id', 'channel', 'blob_id']).reset_index(drop=True)

print(f"\nTotal blobs: {len(df_single)}")
print(df_single.groupby('channel')[['volume_um3','dist_to_surface_um','dist_to_nucleus_um']].describe().round(3))

  measuring cell 1/29 ... 115 blobs
  measuring cell 2/29 ... 351 blobs
  measuring cell 3/29 ... 148 blobs
  measuring cell 4/29 ... 148 blobs
  measuring cell 5/29 ... 168 blobs
  measuring cell 6/29 ... 213 blobs
  measuring cell 7/29 ... 212 blobs
  measuring cell 8/29 ... 90 blobs
  measuring cell 9/29 ... 249 blobs
  measuring cell 10/29 ... 162 blobs
  measuring cell 11/29 ... 152 blobs
  measuring cell 12/29 ... 251 blobs
  measuring cell 13/29 ... 137 blobs
  measuring cell 14/29 ... 206 blobs
  measuring cell 15/29 ... 299 blobs
  measuring cell 16/29 ... 116 blobs
  measuring cell 17/29 ... 263 blobs
  measuring cell 18/29 ... 164 blobs
  measuring cell 19/29 ... 118 blobs
  measuring cell 20/29 ... 158 blobs
  measuring cell 21/29 ... 169 blobs
  measuring cell 22/29 ... 60 blobs
  measuring cell 23/29 ... 32 blobs
  measuring cell 24/29 ... 119 blobs
  measuring cell 25/29 ... 107 blobs
  measuring cell 26/29 ... 117 blobs
  measuring cell 27/29 ... 0 blobs
  measuring cel

In [ ]:
# Save both CSVs for the single test image
df_single, df_single_summary = save_csvs(
    pd.DataFrame(all_rows), out_dir=DATA_DIR, stem=test_file.stem
)
df_single_summary

## 7. Batch Processing

Runs the full pipeline (load → whole-cell segmentation → blob segmentation → measure) on every TIFF in `DATA_DIR`. Results for all files are combined into one CSV.

In [ ]:
from scipy.ndimage import gaussian_filter

OUTPUT_CSV    = DATA_DIR / "EGF_batch_measurements.csv"
SAVE_MASKS    = True    # save cell + blob masks as TIFFs next to each source file
SHOW_NAPARI   = False   # set True to open Napari after each file (single-image mode only)

all_rows_batch = []
tif_files = sorted(DATA_DIR.glob("*.tif")) + sorted(DATA_DIR.glob("*.tiff"))
print(f"Found {len(tif_files)} file(s) in {DATA_DIR}\n")

for file_path in tif_files:
    print(f"── {file_path.name}")

    # ── Load ──────────────────────────────────────────────────────────────
    img   = tifffile.imread(file_path)        # expected shape: (C, Z, Y, X)
    ch0_b = img[0];  ch1_b = img[1]
    ch2_b = img[2];  ch3_b = img[3]

    # ── Whole-cell segmentation (Cellpose) ────────────────────────────────
    def norm_float(arr):
        arr = arr.astype(np.float32)
        arr -= arr.min()
        if arr.max() > 0:
            arr /= arr.max()
        return arr

    ch0_blurred = gaussian_filter(ch0_b.astype(np.float32), sigma=BLUR_SIGMA)
    cyto_nuc_b  = np.stack([norm_float(ch0_blurred), norm_float(ch3_b)], axis=-1)

    masks_b, _, _ = cp_model.eval(
        cyto_nuc_b,
        diameter         = CELL_DIAMETER,
        z_axis           = 0,
        channel_axis     = 3,
        do_3D            = False,
        stitch_threshold = STITCH_THRESHOLD,
        anisotropy       = ANISOTROPY,
    )
    n_cells_b = int(masks_b.max())
    print(f"   {n_cells_b} cell(s) detected")

    # Accumulate full-volume blob label arrays (same shape as source)
    blob_vol_ch0 = np.zeros_like(masks_b)
    blob_vol_ch1 = np.zeros_like(masks_b)
    blob_vol_ch2 = np.zeros_like(masks_b)
    label_offset = 0   # keep blob IDs unique across cells in the volume

    # ── Measure each cell ─────────────────────────────────────────────────
    for cid in range(1, n_cells_b + 1):
        print(f"   cell {cid}/{n_cells_b} ...", end=" ", flush=True)
        try:
            single_mask = (masks_b == cid)
            bbox        = ndi.find_objects(masks_b)[cid - 1]
            mask_crop   = single_mask[bbox]

            bl0 = segment_blobs_3d(ch0_b[bbox], mask_crop, **seg_params)
            bl1 = segment_blobs_3d(ch1_b[bbox], mask_crop, **seg_params)
            bl2 = segment_blobs_3d(ch2_b[bbox], mask_crop, **seg_params)

            # Place blobs back into full-volume arrays (offset labels so each
            # blob has a unique ID across the whole image)
            for vol, bl in [(blob_vol_ch0, bl0),
                            (blob_vol_ch1, bl1),
                            (blob_vol_ch2, bl2)]:
                shifted            = np.where(bl > 0, bl + label_offset, 0)
                vol[bbox]         += shifted
            label_offset += max(bl0.max(), bl1.max(), bl2.max())

            rows = measure_cell(
                file_name  = file_path.name,
                cell_id    = cid,
                cell_masks = masks_b,
                ch0=ch0_b, ch1=ch1_b, ch2=ch2_b, ch3=ch3_b,
                seg_params = seg_params,
            )
            all_rows_batch.extend(rows)
            print(f"{len(rows)} blobs")

        except Exception as e:
            print(f"SKIPPED — {e}")

    # ── Save segmentation masks ────────────────────────────────────────────
    if SAVE_MASKS:
        stem = file_path.stem
        tifffile.imwrite(DATA_DIR / f"{stem}_cell_masks.tif",
                         masks_b.astype(np.uint16))
        tifffile.imwrite(DATA_DIR / f"{stem}_blobs_ch0.tif",
                         blob_vol_ch0.astype(np.uint16))
        tifffile.imwrite(DATA_DIR / f"{stem}_blobs_ch1.tif",
                         blob_vol_ch1.astype(np.uint16))
        tifffile.imwrite(DATA_DIR / f"{stem}_blobs_ch2.tif",
                         blob_vol_ch2.astype(np.uint16))
        print(f"   masks saved → {stem}_cell_masks / _blobs_ch0/1/2.tif")

    # ── Optional Napari preview (leave False during batch) ────────────────
    if SHOW_NAPARI:
        import napari
        v = napari.Viewer(title=file_path.name)
        v.add_image(ch0_b, name="Ch0", colormap="green",  blending="additive")
        v.add_image(ch1_b, name="Ch1", colormap="cyan",   blending="additive")
        v.add_image(ch2_b, name="Ch2", colormap="magenta",blending="additive")
        v.add_image(ch3_b, name="Ch3 nucleus", colormap="blue", blending="additive")
        v.add_labels(masks_b,     name="whole-cell masks")
        v.add_labels(blob_vol_ch0, name="blobs Ch0")
        v.add_labels(blob_vol_ch1, name="blobs Ch1")
        v.add_labels(blob_vol_ch2, name="blobs Ch2")

    print()

# ── Combine and save CSV ──────────────────────────────────────────────────────
df_batch, df_batch_summary = save_csvs(
    pd.DataFrame(all_rows_batch), out_dir=DATA_DIR, stem="EGF_batch"
)
n_files = df_batch["file"].nunique()
print(f"Done - {len(df_batch):,} blob measurements across {n_files} file(s)")